# Step 1 — Verifikasi Dataset IndoToxic2024

Notebook ini **hanya melakukan inspeksi** (read-only) untuk membuktikan klaim-klaim di `Dataset/resumedata.md`. Tidak ada data yang diubah, disalin, atau diproses lebih lanjut.

Klaim yang akan dibuktikan:
1. Daftar file dataset (nama, ukuran, format)
2. Struktur/schema + contoh data mentah tiap file
3. Jumlah total baris per file
4. **Tepat 2 baris dengan `text` kosong** (ternyata berisi spasi, bukan null)
5. Struktur anotasi multi-anotator (43.692 baris → 28.449 teks unik)
6. Distribusi label biner (0/1)
7. Kolom `topic` multi-label + nilai anomali (`"1"`, `"UNKNOWN"`)
8. Kesesuaian ID anotator antara 2 file

In [1]:
import os
import json
from collections import Counter

# Cari folder Dataset (relatif terhadap lokasi notebook)
candidates = ["Dataset", os.path.join("..", "Dataset")]
DATASET_DIR = next(p for p in candidates if os.path.isdir(p))
print("Folder dataset:", os.path.abspath(DATASET_DIR))

for f in sorted(os.listdir(DATASET_DIR)):
    path = os.path.join(DATASET_DIR, f)
    size = os.path.getsize(path)
    ext = os.path.splitext(f)[1]
    print(f"{f:45s} | {size:>12,} byte | format: {ext}")

Folder dataset: C:\SEMESTER 5\Project Sistem Cerdas\Projek\Dataset
indotoxic2024_annotated_data-3.jsonl          |   34,671,576 byte | format: .jsonl
indotoxic2024_annotator_data.jsonl            |        4,794 byte | format: .jsonl


## 1. Muat kedua file (JSONL = 1 objek JSON per baris)

In [2]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

annotated = load_jsonl(os.path.join(DATASET_DIR, "indotoxic2024_annotated_data-3.jsonl"))
annotators = load_jsonl(os.path.join(DATASET_DIR, "indotoxic2024_annotator_data.jsonl"))

print(f"indotoxic2024_annotated_data-3.jsonl : {len(annotated):,} baris")
print(f"indotoxic2024_annotator_data.jsonl   : {len(annotators):,} baris")

indotoxic2024_annotated_data-3.jsonl : 43,692 baris
indotoxic2024_annotator_data.jsonl   : 19 baris


## 2. Schema + contoh data mentah

### 2a. File anotasi utama

In [3]:
print("KEYS:", list(annotated[0].keys()))
print("\n--- 3 baris mentah pertama ---")
with open(os.path.join(DATASET_DIR, "indotoxic2024_annotated_data-3.jsonl"), encoding="utf-8") as f:
    for line in [next(f) for _ in range(3)]:
        print(line.rstrip("\n"))

KEYS: ['batch_id', 'batch_text_id', 'text_id', 'metadata_id', 'annotator_id', 'text', 'initial_paragraph', 'topic', 'is_noise_or_spam_text', 'related_to_election_2024', 'toxicity', 'profanity_obscenity', 'threat_incitement_to_violence', 'insults', 'identity_attack', 'sexually_explicit']

--- 3 baris mentah pertama ---
{"batch_id": "2", "batch_text_id": "1", "text_id": "2-1", "metadata_id": "123632", "annotator_id": "20", "text": "Kemaren mas sepupuku tegang bgt dari awal. Padahal pagi buta dia yg masih sarungan mukanya sumringah bgt, pas akad ga ada ekspresi samsek. Baru senyum lebar bgt pas disuruh nunjukin buku nikah", "initial_paragraph": "", "topic": "Disabilitas", "is_noise_or_spam_text": 0, "related_to_election_2024": 0, "toxicity": 0, "profanity_obscenity": 0, "threat_incitement_to_violence": 0, "insults": 0, "identity_attack": 0, "sexually_explicit": 0}
{"batch_id": "2", "batch_text_id": "2", "text_id": "2-2", "metadata_id": "265496", "annotator_id": "20", "text": "Yesaya 45:25

### 2b. File profil anotator

In [4]:
print("KEYS:", list(annotators[0].keys()))
print("\n--- 3 baris mentah pertama ---")
with open(os.path.join(DATASET_DIR, "indotoxic2024_annotator_data.jsonl"), encoding="utf-8") as f:
    for line in [next(f) for _ in range(3)]:
        print(line.rstrip("\n"))

KEYS: ['annotator_id', 'ethnicity', 'religion', 'disability', 'lgbt', 'gender', 'age', 'city', 'last_education_degree', 'job_status', 'president vote leaning']

--- 3 baris mentah pertama ---
{"annotator_id": 1, "ethnicity": "Tionghoa", "religion": "Kristen", "disability": "no", "lgbt": "no", "gender": "F", "age": 50, "city": "Jakarta", "last_education_degree": "Diploma", "job_status": "Bekerja", "president vote leaning": "2"}
{"annotator_id": 2, "ethnicity": "Madura", "religion": "Islam", "disability": "no", "lgbt": "no", "gender": "F", "age": 22, "city": "Madura", "last_education_degree": "Sekolah Menengah Atas (SMA)", "job_status": "Pelajar/Mahasiswa", "president vote leaning": "3"}
{"annotator_id": 3, "ethnicity": "Batak", "religion": "Kepercayaan Lokal", "disability": "no", "lgbt": "no", "gender": "F", "age": 29, "city": "Medan", "last_education_degree": "Sarjana (S1)", "job_status": "Bekerja", "president vote leaning": "3"}


## 3. BUKTI: baris dengan `text` kosong

Klaim: hanya ada **2 baris** yang `text`-nya kosong, di baris fisik **1256** dan **28394**, dan isinya **spasi** (bukan null/kosong murni).

In [5]:
empty_text_rows = []
with open(os.path.join(DATASET_DIR, "indotoxic2024_annotated_data-3.jsonl"), encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        obj = json.loads(line)
        if obj["text"].strip() == "":
            empty_text_rows.append((line_no, obj))

print(f"Jumlah baris dengan text kosong: {len(empty_text_rows)}\n")
for line_no, obj in empty_text_rows:
    print(f"Baris fisik ke-{line_no}")
    print(f"  text_id      : {obj['text_id']}")
    print(f"  annotator_id : {obj['annotator_id']}")
    print(f"  text (repr)  : {obj['text']!r}   <-- hanya spasi, bukan null")
    print(f"  topic        : {obj['topic']}")
    print()

Jumlah baris dengan text kosong: 2

Baris fisik ke-1256
  text_id      : 2-1256
  annotator_id : 21
  text (repr)  : ' '   <-- hanya spasi, bukan null
  topic        : UNKNOWN

Baris fisik ke-28394
  text_id      : 103-150
  annotator_id : 6
  text (repr)  : '  '   <-- hanya spasi, bukan null
  topic        : UNKNOWN



## 4. Struktur anotasi: 1 baris = 1 anotasi (bukan 1 teks)

Klaim: 43.692 baris → **28.449 teks unik**; sebagian teks dinilai hingga 13 anotator.

In [6]:
text_id_counts = Counter(o["text_id"] for o in annotated)
per_text_dist = Counter(text_id_counts.values())

print(f"Total baris anotasi        : {len(annotated):,}")
print(f"Teks unik (text_id)        : {len(text_id_counts):,}")
print(f"metadata_id unik           : {len({o['metadata_id'] for o in annotated}):,}")
print("\nDistribusi jumlah anotasi per teks:")
print(f"{'anotasi/teks':>12s} | {'jumlah teks':>11s}")
for n_ann, n_text in sorted(per_text_dist.items()):
    print(f"{n_ann:>12d} | {n_text:>11,}")

# Sanity check: total harus sama dengan jumlah baris
assert sum(n_ann * n_text for n_ann, n_text in per_text_dist.items()) == len(annotated)
print("\n[OK] Konsisten: total = jumlah baris")

Total baris anotasi        : 43,692
Teks unik (text_id)        : 28,449
metadata_id unik           : 24,895

Distribusi jumlah anotasi per teks:
anotasi/teks | jumlah teks
           1 |      18,039
           2 |       9,864
           3 |          96
           5 |           1
          10 |           5
          11 |          95
          13 |         349

[OK] Konsisten: total = jumlah baris


## 5. Distribusi label (semua biner 0/1)

Klaim: `toxicity` = label utama, positif ~15,8% (6.899 dari 43.692).

In [7]:
label_cols = [
    "toxicity", "profanity_obscenity", "threat_incitement_to_violence",
    "insults", "identity_attack", "sexually_explicit",
    "is_noise_or_spam_text", "related_to_election_2024",
]
total = len(annotated)
print(f"{'kolom':32s} {'0':>7s} {'1':>7s} {'%1':>7s}")
print("-" * 56)
for col in label_cols:
    c = Counter(o[col] for o in annotated)
    n1 = c.get(1, 0)
    print(f"{col:32s} {c.get(0, 0):>7,} {n1:>7,} {n1/total:>6.1%}")

# Pastikan tidak ada nilai selain 0/1
for col in label_cols:
    assert set(Counter(o[col] for o in annotated)) <= {0, 1}, col
print("\n[OK] Semua kolom label benar-benar biner (hanya 0/1)")

kolom                                  0       1      %1
--------------------------------------------------------
toxicity                          36,793   6,899  15.8%
profanity_obscenity               42,398   1,294   3.0%
threat_incitement_to_violence     42,188   1,504   3.4%
insults                           40,470   3,222   7.4%
identity_attack                   40,495   3,197   7.3%
sexually_explicit                 43,457     235   0.5%
is_noise_or_spam_text             40,755   2,937   6.7%
related_to_election_2024          38,870   4,822  11.0%

[OK] Semua kolom label benar-benar biner (hanya 0/1)


## 6. Kolom `topic`: multi-label (comma-separated) + nilai anomali

In [8]:
topic_counter = Counter()
for o in annotated:
    for t in o["topic"].split(","):
        topic_counter[t.strip()] += 1

print(f"Jumlah topik unik: {len(topic_counter)}\n")
for t, n in topic_counter.most_common():
    print(f"{t:15s} {n:>7,}")

print('\n[!] Nilai anomali: "1" dan "UNKNOWN" — bukan nama topik valid')

Jumlah topik unik: 11

Jewish           13,013
Terpolarisasi    10,196
Disabilitas      10,058
Tionghoa          8,162
UNKNOWN           3,912
Kristen           3,350
Rohingya          1,902
LGBTQ+            1,750
Syiah             1,287
Ahmadiyah           474
1                     5

[!] Nilai anomali: "1" dan "UNKNOWN" — bukan nama topik valid


## 7. Konsistensi antar-file: ID anotator

In [9]:
ids_profile = {a["annotator_id"] for a in annotators}
ids_data = {int(o["annotator_id"]) for o in annotated}

print(f"ID di file profil ({len(ids_profile)}) : {sorted(ids_profile)}")
print(f"ID dipakai di data ({len(ids_data)}) : {sorted(ids_data)}")
print(f"Sama persis? {ids_profile == ids_data}")
print(f"ID yang tidak ada (14, 17)? {sorted({14, 17} - ids_profile)}")

print("\nGender:", dict(Counter(a["gender"] for a in annotators)))
print("Rentang usia:", min(a["age"] for a in annotators), "-", max(a["age"] for a in annotators))

ID di file profil (19) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 18, 19, 20, 21]
ID dipakai di data (19) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 18, 19, 20, 21]
Sama persis? True
ID yang tidak ada (14, 17)? [14, 17]

Gender: {'F': 12, 'M': 7}
Rentang usia: 19 - 50


## Kesimpulan

Semua klaim di `resumedata.md` **terbukti** secara terprogram:

| # | Klaim | Status |
|---|---|---|
| 1 | 2 file JSONL (~34,6 MB + ~4,8 KB) | ✔ |
| 2 | 43.692 baris anotasi, 19 baris profil anotator | ✔ |
| 3 | **2 baris `text` kosong** = baris fisik 1256 & 28394, isinya spasi | ✔ |
| 4 | 28.449 teks unik, hingga 13 anotator per teks | ✔ |
| 5 | Label biner 0/1, toxicity positif 15,8% | ✔ |
| 6 | `topic` multi-label, ada anomali `"1"` & `"UNKNOWN"` | ✔ |
| 7 | 19 anotator konsisten antar-file (tanpa ID 14 & 17) | ✔ |

> Notebook ini read-only. Tidak ada pemrosesan data (cleaning/agregasi) yang dilakukan — menunggu konfirmasi untuk Step 2.